# 3. DataRobot Tools

*Estimated time to run notebook: about 8-10 min*

**Goal:** Empower the agent to query enterprise data warehouses (like Snowflake) to answer factual business questions.

**Key Concept:** This setup lets the agent dynamically generate queries and retrieve live datasets—transforming it from a simple chatbot into a data analyst capable of answering questions like *"What is the average dam_price_usd_mwh for all hubs?"*

**Note:** Use an existing deployed MCP server, or create one in **Notebook 0 - MCP Server Setup (Optional)**. MCP server setup is not repeated in this notebook.

---

### Agent context: how MCP meets the agent

In this workshop, an **agent** is an LLM that can **reason in steps** and **call tools** when it needs facts or actions it cannot invent. The model proposes tool calls; the runtime executes them and returns results; the model continues until it can answer the user. That loop is what turns a chatbot into something that can act on real data.

**MCP (Model Context Protocol)** is the **wire format and discovery layer** for those tools: a deployment exposes a list of tool names, schemas, and handlers over HTTP. The agent does not talk to Snowflake (or your warehouse) directly—it calls MCP tools, and **DataRobot** runs the tool implementation behind the deployment URL you configured in Notebook 0.

**In code:** `MCPServerStreamableHTTP(...)` registers that deployment as a **toolset**. `Agent(model=..., toolsets=[server], ...)` gives the LLM access to every tool the MCP server advertises, using your LLM Gateway model for reasoning. So: **agent = model + tool-calling loop**, **MCP = how tools are exposed**, **DataRobot deployment = where tools execute**.

In [1]:
import os
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Load configuration
load_dotenv(override=True)

# 2. Initialize client
dr_client = dr.Client()

# 3. Configure the tool connection (MCP)
# This connects to a DataRobot deployment exposing data tools via MCP.
MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")
if not MCP_DEPLOYMENT_ID:
    raise ValueError(
        "MCP_DEPLOYMENT_ID is not set. Add an existing MCP deployment ID to .env, "
        "or create one in Notebook 0."
    )

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    max_retries=3,
    timeout=60.0,
)

# Display the full list of tools available in the MCP server
print("Available MCP server tools:")
async with server:
    tools = await server.list_tools()

for tool in tools:
    print(tool.name)

    # Uncomment to see the tool description
    # print(tool.description)

ModuleNotFoundError: No module named 'pydantic_ai'


### What happens when you run `agent.run(...)`

1. The **LLM** reads your question and system prompt.
2. If it needs data, it chooses an **MCP tool** (name + arguments) from the tools listed above.
3. **Pydantic AI** sends that call to the **streamable HTTP MCP endpoint** on your DataRobot deployment (same URL as `MCPServerStreamableHTTP`).
4. The **deployment** runs the tool (e.g. query execution) and returns structured results.
5. The **LLM** may call more tools or, when satisfied, replies with a natural-language answer.

So MCP is not “the agent”—it is the **bridge** that lets the agent use enterprise-safe, deployment-hosted tools without embedding credentials or SQL in the notebook.

In [2]:
# 4. Configure the model (via DataRobot LLM Gateway)
MODEL_NAME = os.getenv("MODEL_NAME")
llm = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 5. Define the agent's system prompt
system_prompt = """
Use the ERCOT training dataset, available in the DataRobot AI catalog, to answer user questions.
"""

# 6. Define the agent (LLM + MCP tools + system prompt)
agent = Agent(model=llm, toolsets=[server], system_prompt=system_prompt)

# 7. Execution (example question)
async with server:
    response = await agent.run("What is the average dam_price_usd_mwh for all hubs?")
    # Expected output: $32.84 per MWh
    pprint(response.output)

In [3]:
# 8. (Optional) Test the agent against a specific dataset ID
# If your MCP tools support it, passing the dataset ID can scope queries.
DATASET_ID = os.getenv("ERCOT_TRAINING_DATASET_ID")

async with server:
    response = await agent.run(f"How many unique hub_name are in dataset {DATASET_ID}?")
    # Expected output: 4
    pprint(response.output)